# 🔥 AIOps — Lab 3a
# Install Prometheus and Alertmanager on Kubernetes

---

| | |
|---|---|
| **Course** | AIOps Engineering |
| **Lab** | Lab 3a (v2 — NFS-robust) |
| **Topics** | NFS Setup · Prometheus · Alertmanager · Grafana · Kubernetes Monitoring Stack |
| **Duration** | ~90 minutes |
| **Platform** | Rocky Linux 8.7 · Kubernetes cluster · kubectl configured |

---

## 🎯 Objectives

1. **Detect and auto-configure** the local NFS server on the Rocky Linux node
2. **Install and verify NFS** before any Kubernetes resources are created
3. **Install Prometheus** with NFS-backed persistent storage
4. **Install Alertmanager** with email and Slack routing
5. **Install Grafana** pre-provisioned with Prometheus as data source
6. **Verify** the complete monitoring stack end-to-end

---

## 🗺️ Architecture

```
 Rocky Linux 8.7 Node
 ┌──────────────────────────────────────────────────────────────┐
 │  NFS Server (/mnt/nfs/promdata)  ◀── auto-detected IP       │
 │       │                                                      │
 │       │ PersistentVolume (NFS)                               │
 │  ┌────▼─────────────────────────────────────────────────┐   │
 │  │              monitoring  namespace                   │   │
 │  │                                                      │   │
 │  │  Prometheus :9090  ──scrape──▶  K8s API / cAdvisor  │   │
 │  │       │ alerts                  Nodes / Pods / Svcs  │   │
 │  │  Alertmanager :9093             Grafana :3000        │   │
 │  │       │ email/Slack                                  │   │
 │  └──────────────────────────────────────────────────────┘   │
 └──────────────────────────────────────────────────────────────┘
```

---

## 📋 Sequences

| Seq | Title | What's New |
|-----|-------|------------|
| **0** | Environment Setup | Helpers + kubectl check |
| **1** | Auto-detect Node IP | Finds real non-localhost IP |
| **2** | NFS Install & Configure | Full NFS setup with verification |
| **3** | PV and PVC | NFS-backed storage with readiness check |
| **4** | RBAC | ClusterRole, ServiceAccount, Bindings |
| **5** | ConfigMaps | Prometheus, Alertmanager, Grafana configs |
| **6** | Deployments | All three components |
| **7** | Services | LoadBalancer + NodePort exposure |
| **8** | Verify Stack | Targets, alerts, Grafana health |

> ▶️ **Run each cell in order using `Shift + Enter`**

> 💡 This notebook auto-detects your node IP and NFS server status —
> no manual IP configuration needed.

---
## Sequence 0 — Environment Setup

**Purpose:** Import libraries, define shell helpers, and confirm `kubectl` can
reach the cluster. All later cells depend on the `run()` and `write_yaml()` helpers
defined here.

> ⏱️ Run this cell first every time you open the notebook.

In [1]:
import subprocess, os, time, json as _json, socket, re
import urllib.request

def run(cmd, ignore_errors=False, show_cmd=True):
    if show_cmd:
        print(f'$ {cmd}')
        print('-' * 60)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip() and not ignore_errors:
        print('STDERR:', result.stderr.strip())
    if result.returncode != 0 and not ignore_errors:
        print(f'  [exit code: {result.returncode}]')
    return result.stdout.strip(), result.returncode

def run_q(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.returncode

def write_yaml(path, content):
    with open(path, 'w') as f:
        f.write(content)
    lines = len(content.splitlines())
    print(f'  ✍️  wrote {path} ({lines} lines)')

def section(title):
    print()
    print('=' * 60)
    print(f'  {title}')
    print('=' * 60)

print('✅ Helpers loaded')
section('kubectl connectivity check')
run('kubectl version --client --short 2>/dev/null || kubectl version --client')
print()
run('kubectl get nodes')

✅ Helpers loaded

  kubectl connectivity check
$ kubectl version --client --short 2>/dev/null || kubectl version --client
------------------------------------------------------------
Client Version: v1.35.3
Kustomize Version: v5.7.1

$ kubectl get nodes
------------------------------------------------------------
NAME         STATUS   ROLES                       AGE   VERSION
aiops-labs   Ready    control-plane,etcd,master   83m   v1.23.9+k3s1


('NAME         STATUS   ROLES                       AGE   VERSION\naiops-labs   Ready    control-plane,etcd,master   83m   v1.23.9+k3s1',
 0)

---
## Sequence 1 — Auto-detect Node IP

**Purpose:** Find the real non-localhost IP address of this machine.
This IP is used as the NFS server address in the PersistentVolume manifest
and as the base URL for accessing Prometheus, Alertmanager, and Grafana.

The cell tries three detection strategies in order:
1. Ask `kubectl` for the cluster node's `InternalIP`
2. Inspect active network interfaces (skips `lo` and `docker`/`cni` bridges)
3. Use a UDP socket trick to find the default-route interface IP

> 💡 You can override `NODE_IP` manually in the last line if auto-detection
> picks the wrong interface.

In [2]:
# Sequence 1: Auto-detect the node's non-localhost IP address

def detect_node_ip():
    # Strategy 1: ask kubectl for the node InternalIP
    out, rc = run_q(
        "kubectl get nodes -o jsonpath='{.items[0].status.addresses[?(@.type==\"InternalIP\")].address}'"
    )
    if rc == 0 and out.strip() and not out.strip().startswith('127.'):
        return out.strip(), 'kubectl InternalIP'

    # Strategy 2: scan network interfaces
    out2, _ = run_q("ip -4 addr show | grep 'inet ' | awk '{print $2}' | cut -d/ -f1")
    for ip in out2.splitlines():
        ip = ip.strip()
        if ip and not ip.startswith('127.') and not ip.startswith('172.') \
               and not ip.startswith('10.42.') and not ip.startswith('10.43.'):
            return ip, 'network interface scan'

    # Strategy 3: UDP socket trick (doesn't send any data)
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(('8.8.8.8', 80))
        ip = s.getsockname()[0]
        s.close()
        if not ip.startswith('127.'):
            return ip, 'UDP socket default route'
    except Exception:
        pass

    return '10.10.0.100', 'fallback (manual update required)'

NODE_IP, detection_method = detect_node_ip()

section('Node IP Detection')
print(f'  Detection method : {detection_method}')
print(f'  NODE_IP          : {NODE_IP}')

# Show all IPs for reference
print()
print('All detected IP addresses on this host:')
run('ip -4 addr show | grep inet')

print()
print('If NODE_IP is wrong, override it here:')
print(f'  NODE_IP = \'<correct-ip>\'  # then re-run this cell')
print()
print(f'✅ Using NODE_IP = {NODE_IP}')


  Node IP Detection
  Detection method : kubectl InternalIP
  NODE_IP          : 10.0.0.95

All detected IP addresses on this host:
$ ip -4 addr show | grep inet
------------------------------------------------------------
inet 127.0.0.1/8 scope host lo
    inet 10.0.0.95/24 brd 10.0.0.255 scope global dynamic noprefixroute ens3
    inet 10.42.0.0/32 scope global flannel.1
    inet 10.42.0.1/24 brd 10.42.0.255 scope global cni0

If NODE_IP is wrong, override it here:
  NODE_IP = '<correct-ip>'  # then re-run this cell

✅ Using NODE_IP = 10.0.0.95


---
## Sequence 2 — NFS Server: Install, Configure, and Verify

**Purpose:** Ensure the NFS server is fully operational before creating any
Kubernetes storage resources. The cell below:

1. **Checks** if `nfs-utils` is installed — installs it if missing
2. **Creates** the Prometheus data directory with correct ownership and permissions
3. **Configures** `/etc/exports` with the correct NFS export entry
4. **Starts and enables** the NFS server service
5. **Opens** firewall ports for NFS, mountd, and rpc-bind
6. **Verifies** the export is visible via `showmount`
7. **Reports** a clear pass/fail result — the lab stops here if NFS fails

### 📖 Why NFS for Prometheus?

Prometheus stores time-series data in its TSDB (Time Series Database) on disk.
Without persistent storage, all collected metrics are lost every time the pod
restarts. NFS provides a network-attached volume that survives pod restarts,
node drains, and redeployments.

```
 Prometheus pod  ──mount──▶  PVC (pvc-nfs-data)
                                 │
                             PV (pv-nfs-data)
                                 │
                             NFS export
                             /mnt/nfs/promdata
                                 │
                          Rocky Linux NFS Server
                          (this machine: NODE_IP)
```

> ⚠️ **This cell uses `sudo`** for NFS installation and firewall configuration.
> It must run on the same machine that serves as both the NFS server and the
> cluster node (typical single-node lab setup).

In [3]:
# Sequence 2: Full NFS install, configure, and verify
NFS_EXPORT_DIR = '/mnt/nfs/promdata'

section('Step 2.1 — Check / Install nfs-utils')
_, nfs_installed = run_q('rpm -q nfs-utils')
if nfs_installed == 0:
    print('  ✅ nfs-utils already installed')
else:
    print('  📦 Installing nfs-utils...')
    run('sudo dnf install -y nfs-utils')

section('Step 2.2 — Create NFS export directory')
run(f'sudo mkdir -p {NFS_EXPORT_DIR}')
run(f'sudo chown nobody:nobody {NFS_EXPORT_DIR}')
run(f'sudo chmod 777 {NFS_EXPORT_DIR}')
run(f'ls -ld {NFS_EXPORT_DIR}')

section('Step 2.3 — Configure /etc/exports')
export_line = f'{NFS_EXPORT_DIR} {NODE_IP}(rw,sync,no_subtree_check,no_root_squash)'
# Read current exports
try:
    with open('/etc/exports') as f:
        current_exports = f.read()
except FileNotFoundError:
    current_exports = ''

if NFS_EXPORT_DIR in current_exports:
    print(f'  ✅ Export entry already present in /etc/exports')
else:
    print(f'  Adding export: {export_line}')
    run(f'echo "{export_line}" | sudo tee -a /etc/exports')

print('  Current /etc/exports:')
run('cat /etc/exports')

section('Step 2.4 — Enable and start NFS server')
run('sudo systemctl enable --now nfs-server')
time.sleep(2)
out, _ = run_q('systemctl is-active nfs-server')
if out.strip() == 'active':
    print('  ✅ nfs-server is active')
else:
    print(f'  ❌ nfs-server status: {out} — check logs with: journalctl -u nfs-server -n 20')

section('Step 2.5 — Reload NFS exports')
run('sudo exportfs -ra')
run('sudo exportfs -v')

section('Step 2.6 — Open firewall ports')
_, fw_active = run_q('systemctl is-active firewalld')
if fw_active == 0:
    run('sudo firewall-cmd --permanent --add-service=nfs')
    run('sudo firewall-cmd --permanent --add-service=mountd')
    run('sudo firewall-cmd --permanent --add-service=rpc-bind')
    run('sudo firewall-cmd --reload')
    print('  ✅ Firewall rules applied')
else:
    print('  ℹ️  firewalld not active — skipping firewall configuration')

section('Step 2.7 — Verify export is visible (showmount)')
out, rc = run(f'showmount -e {NODE_IP}')
if rc == 0 and NFS_EXPORT_DIR in out:
    print(f'  ✅ NFS export visible: {NFS_EXPORT_DIR}')
    NFS_READY = True
else:
    print(f'  ❌ showmount could not see the export.')
    print('     Possible causes:')
    print('     - rpcbind not running: sudo systemctl start rpcbind')
    print('     - firewall blocking: sudo firewall-cmd --add-service=rpc-bind')
    print('     - wrong IP in /etc/exports')
    NFS_READY = False

section('NFS Readiness Summary')
if NFS_READY:
    print('  ✅ NFS server is ready')
    print(f'  Server IP  : {NODE_IP}')
    print(f'  Export path: {NFS_EXPORT_DIR}')
    print('  ▶️  Continue to Sequence 3')
else:
    print('  ❌ NFS is NOT ready — fix the issues above before continuing')
    print('  🛑 DO NOT proceed to Sequence 3 until NFS_READY = True')
    raise SystemExit('NFS not ready — stopping notebook execution')


  Step 2.1 — Check / Install nfs-utils
  ✅ nfs-utils already installed

  Step 2.2 — Create NFS export directory
$ sudo mkdir -p /mnt/nfs/promdata
------------------------------------------------------------
$ sudo chown nobody:nobody /mnt/nfs/promdata
------------------------------------------------------------
$ sudo chmod 777 /mnt/nfs/promdata
------------------------------------------------------------
$ ls -ld /mnt/nfs/promdata
------------------------------------------------------------
drwxrwxrwx 2 nobody nobody 6 Apr 16 03:46 /mnt/nfs/promdata

  Step 2.3 — Configure /etc/exports
  ✅ Export entry already present in /etc/exports
  Current /etc/exports:
$ cat /etc/exports
------------------------------------------------------------
/mnt/nfs/promdata 10.10.0.100(rw,sync,no_subtree_check)

  Step 2.4 — Enable and start NFS server
$ sudo systemctl enable --now nfs-server
------------------------------------------------------------
  ✅ nfs-server is active

  Step 2.5 — Reload NFS e

### Step 2.8 — Test NFS mount from a client

This optional step confirms that the NFS export can actually be mounted
by a client — exactly what the Kubernetes kubelet will do when attaching
the PersistentVolume to the Prometheus pod.

> This cell uses a temporary mount point and cleans up after itself.

In [4]:
# Sequence 2, Step 2.8: Optional — test mount and unmount
TEST_MOUNT = '/tmp/nfs_test_mount'

section('Step 2.8 — Test NFS mount from client')
run(f'sudo mkdir -p {TEST_MOUNT}')

_, rc_mount = run(f'sudo mount -t nfs {NODE_IP}:{NFS_EXPORT_DIR} {TEST_MOUNT}',
                  ignore_errors=True)
if rc_mount == 0:
    print(f'  ✅ NFS mount succeeded at {TEST_MOUNT}')
    # Write a test file
    run(f'echo "nfs-test-$(date)" | sudo tee {TEST_MOUNT}/test.txt > /dev/null')
    run(f'ls -la {TEST_MOUNT}')
    # Cleanup
    run(f'sudo umount {TEST_MOUNT}')
    run(f'sudo rm -rf {TEST_MOUNT}')
    print('  ✅ Mount test passed — NFS read/write confirmed')
    print('  ✅ Prometheus PVC will bind successfully')
else:
    print('  ⚠️  NFS mount test failed.')
    print('     This may still work if the kubelet has different permissions.')
    print('     Check: sudo journalctl -u nfs-server -n 30')
    run(f'sudo rm -rf {TEST_MOUNT}', show_cmd=False)


  Step 2.8 — Test NFS mount from client
$ sudo mkdir -p /tmp/nfs_test_mount
------------------------------------------------------------
$ sudo mount -t nfs 10.0.0.95:/mnt/nfs/promdata /tmp/nfs_test_mount
------------------------------------------------------------
  ⚠️  NFS mount test failed.
     This may still work if the kubelet has different permissions.
     Check: sudo journalctl -u nfs-server -n 30


---
## Sequence 3 — Create Monitoring Namespace

All monitoring resources are isolated in the `monitoring` namespace.
This provides better access control, resource quotas, and keeps monitoring
components separate from application workloads.

```bash
kubectl create namespace monitoring
```

In [5]:
# Sequence 3: Create the monitoring namespace
section('Sequence 3: Create monitoring namespace')
out, rc = run('kubectl create namespace monitoring 2>/dev/null', ignore_errors=True)
if rc == 0:
    print('  ✅ Namespace created')
else:
    print('  ℹ️  Namespace already exists — continuing')
print()
run('kubectl get ns')


  Sequence 3: Create monitoring namespace
$ kubectl create namespace monitoring 2>/dev/null
------------------------------------------------------------
namespace/monitoring created
  ✅ Namespace created

$ kubectl get ns
------------------------------------------------------------
NAME                   STATUS   AGE
default                Active   83m
kube-node-lease        Active   83m
kube-public            Active   83m
kube-system            Active   83m
kubernetes-dashboard   Active   64m
monitoring             Active   0s


('NAME                   STATUS   AGE\ndefault                Active   83m\nkube-node-lease        Active   83m\nkube-public            Active   83m\nkube-system            Active   83m\nkubernetes-dashboard   Active   64m\nmonitoring             Active   0s',
 0)

---
## Sequence 4 — Persistent Volume and Persistent Volume Claim

**Purpose:** Create the Kubernetes storage objects that back Prometheus's
TSDB with the NFS share we configured in Sequence 2.

### 📖 PV vs PVC

| Object | Who creates it | What it represents |
|--------|---------------|--------------------|
| **PersistentVolume (PV)** | Admin (this notebook) | The actual storage resource — NFS share |
| **PersistentVolumeClaim (PVC)** | Admin (this notebook) | A request for storage — binds to the PV |
| **Pod volume mount** | Prometheus deployment | Attaches the PVC into the container |

The NFS server IP is taken from `NODE_IP` detected in Sequence 1 — no manual
editing needed.

> The cell waits up to 60 seconds for the PVC to bind and reports clearly
> if it remains `Pending` (which means NFS is unreachable).

In [6]:
# Sequence 4: Create PV and PVC using the auto-detected NODE_IP
NFS_EXPORT_DIR = '/mnt/nfs/promdata'  # must match Sequence 2

pv_pvc_lines = [
    '---',
    'apiVersion: v1',
    'kind: PersistentVolume',
    'metadata:',
    '  name: pv-nfs-data',
    '  namespace: monitoring',
    '  labels:',
    '    type: nfs',
    '    app: prometheus-deployment',
    'spec:',
    '  storageClassName: managed-nfs',
    '  capacity:',
    '    storage: 1Gi',
    '  accessModes:',
    '    - ReadWriteMany',
    '  nfs:',
    f'    server: {NODE_IP}',
    f'    path: "{NFS_EXPORT_DIR}"',
    '---',
    'apiVersion: v1',
    'kind: PersistentVolumeClaim',
    'metadata:',
    '  name: pvc-nfs-data',
    '  namespace: monitoring',
    '  labels:',
    '    app: prometheus-deployment',
    'spec:',
    '  storageClassName: managed-nfs',
    '  accessModes:',
    '    - ReadWriteMany',
    '  resources:',
    '    requests:',
    '      storage: 500Mi',
]
write_yaml('/tmp/pv-pvc.yaml', '\n'.join(pv_pvc_lines) + '\n')

section('Applying PV and PVC')
print(f'  NFS server : {NODE_IP}')
print(f'  NFS path   : {NFS_EXPORT_DIR}')
print()
run('kubectl apply -f /tmp/pv-pvc.yaml')

# Wait for PVC to bind (up to 60 seconds)
section('Waiting for PVC to bind')
bound = False
for attempt in range(12):
    pvc_out, _ = run_q('kubectl get pvc pvc-nfs-data -n monitoring -o jsonpath="{.status.phase}"')
    pv_out, _  = run_q('kubectl get pv pv-nfs-data -o jsonpath="{.status.phase}" 2>/dev/null')
    print(f'  [{attempt+1:02d}/12] PVC: {pvc_out:<12} PV: {pv_out}')
    if pvc_out == 'Bound':
        bound = True
        break
    time.sleep(5)

print()
if bound:
    print('  ✅ PVC is Bound — Prometheus storage ready')
else:
    print('  ⚠️  PVC is still Pending after 60s')
    print('  Possible causes:')
    print('    - NFS server not reachable from kubelet')
    print('    - storageClassName mismatch (expected: managed-nfs)')
    print('    - NFS export path not accessible')
    print()
    print('  Run these to diagnose:')
    print(f'    showmount -e {NODE_IP}')
    print('    kubectl describe pvc pvc-nfs-data -n monitoring')
    print('    kubectl describe pv pv-nfs-data')
    print()
    print('  To proceed without NFS (emptyDir fallback), run Sequence 4b below.')

run('kubectl get pv,pvc -n monitoring')

  ✍️  wrote /tmp/pv-pvc.yaml (33 lines)

  Applying PV and PVC
  NFS server : 10.0.0.95
  NFS path   : /mnt/nfs/promdata

$ kubectl apply -f /tmp/pv-pvc.yaml
------------------------------------------------------------
persistentvolumeclaim/pvc-nfs-data created
STDERR: The PersistentVolume "pv-nfs-data" is invalid: spec.persistentvolumesource: Forbidden: spec.persistentvolumesource is immutable after creation
  core.PersistentVolumeSource{
  	... // 2 identical fields
  	HostPath:  nil,
  	Glusterfs: nil,
  	NFS: &core.NFSVolumeSource{
- 		Server:   "10.0.0.95",
+ 		Server:   "10.10.0.100",
  		Path:     "/mnt/nfs/promdata",
  		ReadOnly: false,
  	},
  	RBD:     nil,
  	Quobyte: nil,
  	... // 15 identical fields
  }
  [exit code: 1]

  Waiting for PVC to bind
  [01/12] PVC: Pending      PV: Released
  [02/12] PVC: Pending      PV: Released
  [03/12] PVC: Pending      PV: Released
  [04/12] PVC: Pending      PV: Released
  [05/12] PVC: Pending      PV: Released
  [06/12] PVC: Pending 

('NAME                           CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS     CLAIM                     STORAGECLASS   REASON   AGE\npersistentvolume/pv-nfs-data   1Gi        RWX            Retain           Released   monitoring/pvc-nfs-data   managed-nfs             26m\n\nNAME                                 STATUS    VOLUME   CAPACITY   ACCESS MODES   STORAGECLASS   AGE\npersistentvolumeclaim/pvc-nfs-data   Pending                                      managed-nfs    61s',
 0)

### Sequence 4b — Fallback: Use emptyDir if NFS is unavailable

If the PVC remains `Pending`, this cell patches the Prometheus deployment to
use `emptyDir` instead of NFS. This means metric history is lost on pod restart,
but everything else works normally.

> 💡 Only run this cell if the PVC above shows `Pending` and you want to
> continue the lab without fixing NFS right now.

In [7]:
# Sequence 4b: FALLBACK — switch Prometheus to emptyDir if PVC is Pending
# Only run this if pvc-nfs-data is stuck in Pending state.

pvc_status, _ = run_q('kubectl get pvc pvc-nfs-data -n monitoring -o jsonpath="{.status.phase}" 2>/dev/null')

if pvc_status == 'Bound':
    print('✅ PVC is Bound — emptyDir fallback not needed')
    USE_EMPTY_DIR = False
else:
    print(f'⚠️  PVC status is "{pvc_status}" — activating emptyDir fallback')
    USE_EMPTY_DIR = True
    print()
    print('Prometheus will use emptyDir (non-persistent) storage.')
    print('Metrics history will NOT survive pod restarts.')
    print('Fix NFS later and re-run Sequence 4 + 6 to switch back to NFS.')

print(f'\nUSE_EMPTY_DIR = {USE_EMPTY_DIR}')

⚠️  PVC status is "Pending" — activating emptyDir fallback

Prometheus will use emptyDir (non-persistent) storage.
Metrics history will NOT survive pod restarts.
Fix NFS later and re-run Sequence 4 + 6 to switch back to NFS.

USE_EMPTY_DIR = True


---
## Sequence 5 — RBAC: ClusterRole, ServiceAccount, ClusterRoleBinding

Prometheus needs **cluster-wide read access** to discover and scrape:
nodes, pods, services, endpoints, and API server metrics.

```
 ServiceAccount (prometheus)
        │
        ▼
 ClusterRoleBinding (prometheus)
        │
        ▼
 ClusterRole (prometheus)
   [get, list, watch]
   nodes, nodes/proxy, services,
   endpoints, pods, /metrics
```

In [8]:
# Sequence 5, Step 1: ClusterRoles
section('Sequence 5: RBAC')

cluster_roles = '''---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: prometheus
rules:
- apiGroups: [""]
  resources:
  - nodes
  - nodes/proxy
  - services
  - endpoints
  - pods
  verbs: ["get", "list", "watch"]
- apiGroups:
  - extensions
  resources:
  - ingresses
  verbs: ["get", "list", "watch"]
- nonResourceURLs: ["/metrics"]
  verbs: ["get"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  labels:
    app.kubernetes.io/name: kube-state-metrics
    app.kubernetes.io/version: v1.8.0
  name: kube-state-metrics
rules:
- apiGroups: [""]
  resources: [configmaps,secrets,nodes,pods,services,resourcequotas,
               replicationcontrollers,limitranges,persistentvolumeclaims,
               persistentvolumes,namespaces,endpoints]
  verbs: ["list", "watch"]
- apiGroups: ["extensions"]
  resources: ["daemonsets", "deployments", "replicasets", "ingresses"]
  verbs: ["list", "watch"]
- apiGroups: ["apps"]
  resources: ["statefulsets", "daemonsets", "deployments", "replicasets"]
  verbs: ["list", "watch"]
- apiGroups: ["autoscaling"]
  resources: ["horizontalpodautoscalers"]
  verbs: ["list", "watch"]
- apiGroups: ["authentication.k8s.io"]
  resources: ["tokenreviews"]
  verbs: ["create"]
- apiGroups: ["authorization.k8s.io"]
  resources: ["subjectaccessreviews"]
  verbs: ["create"]
- apiGroups: ["policy"]
  resources: ["poddisruptionbudgets"]
  verbs: ["list", "watch"]
- apiGroups: ["certificates.k8s.io"]
  resources: ["certificatesigningrequests"]
  verbs: ["list", "watch"]
- apiGroups: ["storage.k8s.io"]
  resources: ["storageclasses", "volumeattachments"]
  verbs: ["list", "watch"]
- apiGroups: ["admissionregistration.k8s.io"]
  resources: [mutatingwebhookconfigurations, validatingwebhookconfigurations]
  verbs: ["list", "watch"]
- apiGroups: ["networking.k8s.io"]
  resources: ["networkpolicies"]
  verbs: ["list", "watch"]
'''
write_yaml('/tmp/cluster-roles.yaml', cluster_roles)
run('kubectl apply -f /tmp/cluster-roles.yaml')


  Sequence 5: RBAC
  ✍️  wrote /tmp/cluster-roles.yaml (65 lines)
$ kubectl apply -f /tmp/cluster-roles.yaml
------------------------------------------------------------
clusterrole.rbac.authorization.k8s.io/prometheus unchanged
clusterrole.rbac.authorization.k8s.io/kube-state-metrics unchanged


('clusterrole.rbac.authorization.k8s.io/prometheus unchanged\nclusterrole.rbac.authorization.k8s.io/kube-state-metrics unchanged',
 0)

In [9]:
# Sequence 5, Steps 2 & 3: ServiceAccount + ClusterRoleBindings
sa_yaml = '''---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: prometheus
  namespace: monitoring
'''
write_yaml('/tmp/service-account.yaml', sa_yaml)
run('kubectl apply -f /tmp/service-account.yaml')

role_bindings = '''---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: prometheus
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: prometheus
subjects:
- kind: ServiceAccount
  name: default
  namespace: monitoring
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  labels:
    app.kubernetes.io/name: kube-state-metrics
    app.kubernetes.io/version: v1.8.0
  name: kube-state-metrics
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: kube-state-metrics
subjects:
- kind: ServiceAccount
  name: kube-state-metrics
  namespace: kube-system
'''
write_yaml('/tmp/role-bindings.yaml', role_bindings)
run('kubectl apply -f /tmp/role-bindings.yaml')

print()
print('📋 RBAC summary:')
run('kubectl get clusterrole prometheus kube-state-metrics 2>/dev/null')
run('kubectl get clusterrolebinding prometheus kube-state-metrics 2>/dev/null')
run('kubectl get serviceaccount prometheus -n monitoring 2>/dev/null')

  ✍️  wrote /tmp/service-account.yaml (6 lines)
$ kubectl apply -f /tmp/service-account.yaml
------------------------------------------------------------
serviceaccount/prometheus created
  ✍️  wrote /tmp/role-bindings.yaml (29 lines)
$ kubectl apply -f /tmp/role-bindings.yaml
------------------------------------------------------------
clusterrolebinding.rbac.authorization.k8s.io/prometheus unchanged
clusterrolebinding.rbac.authorization.k8s.io/kube-state-metrics unchanged

📋 RBAC summary:
$ kubectl get clusterrole prometheus kube-state-metrics 2>/dev/null
------------------------------------------------------------
NAME                 CREATED AT
prometheus           2026-04-16T03:40:24Z
kube-state-metrics   2026-04-16T03:40:24Z
$ kubectl get clusterrolebinding prometheus kube-state-metrics 2>/dev/null
------------------------------------------------------------
NAME                 ROLE                             AGE
prometheus           ClusterRole/prometheus           27m
kube-st

('NAME         SECRETS   AGE\nprometheus   1         0s', 0)

---
## Sequence 6 — ConfigMaps

ConfigMaps inject configuration files into pods without rebuilding images.

| ConfigMap | Pod | Purpose |
|-----------|-----|---------|
| `prometheus-server-conf` | Prometheus | `prometheus.yml` scrape config + alert rules |
| `alertmanager-config` | Alertmanager | Routing tree + receiver definitions |
| `alertmanager-templates` | Alertmanager | Go notification templates |
| `grafana-datasources` | Grafana | Auto-provision Prometheus data source |

### 📖 Prometheus Scrape Jobs

| Job | What it scrapes |
|-----|-----------------|
| `prometheus` | Prometheus itself (self-monitoring) |
| `kubernetes-apiservers` | K8s API server via HTTPS + bearer token |
| `kubernetes-nodes` | kubelet proxy — node-level metrics |
| `kubernetes-pods` | Pods with `prometheus.io/scrape: true` annotation |
| `kubernetes-cadvisor` | Container-level CPU/memory via embedded cAdvisor |

In [10]:
# Sequence 6, Step 1: Prometheus ConfigMap
section('Sequence 6: ConfigMaps')

configmap_prometheus = '''---
apiVersion: v1
kind: ConfigMap
metadata:
  name: prometheus-server-conf
  labels:
    name: prometheus-server-conf
  namespace: monitoring
data:
  prometheus.rules: |-
    groups:
    - name: Infrastructure alerts
      rules:
      - alert: High Pod Memory
        expr: sum(container_memory_usage_bytes) > 1
        for: 1m
        labels:
          severity: page
        annotations:
          summary: High Memory Usage
      - alert: Cluster Memory Usage
        expr: sum(container_memory_working_set_bytes{id="/"})/sum(machine_memory_bytes{}) * 100 > 90
        for: 5m
        labels:
          severity: page
        annotations:
          summary: Cluster Memory Usage > 90%
      - alert: Cluster CPU Usage
        expr: sum (rate (container_cpu_usage_seconds_total{id="/"}[5m])) / sum (machine_cpu_cores{}) * 100 > 90
        for: 5m
        labels:
          severity: page
        annotations:
          summary: "Cluster CPU Usage > 90%"
          description: "Cluster CPU on {{$labels.instance}}: {{ $value }}"
      - alert: Pod Memory usage
        expr: sum (container_memory_working_set_bytes{image!="",name=~"^k8s_.*"}) by (container_label_io_kubernetes_pod_name) > 4026531841
        for: 5m
        labels:
          severity: page
        annotations:
          summary: "Pod Memory usage > 90%"
          description: "Pod Memory on {{$labels.instance}}: {{ $value }}"
  prometheus.yml: |-
    global:
      scrape_interval: 5s
      evaluation_interval: 5s
    rule_files:
      - /etc/prometheus/prometheus.rules
    alerting:
      alertmanagers:
      - scheme: http
        static_configs:
        - targets:
          - "alertmanager.monitoring.svc:9093"
    scrape_configs:
      - job_name: 'prometheus'
        static_configs:
        - targets: ['localhost:9090']
      - job_name: 'kubernetes-apiservers'
        kubernetes_sd_configs:
        - role: endpoints
        scheme: https
        tls_config:
          ca_file: /var/run/secrets/kubernetes.io/serviceaccount/ca.crt
          insecure_skip_verify: true
        bearer_token_file: /var/run/secrets/kubernetes.io/serviceaccount/token
        relabel_configs:
        - source_labels: [__meta_kubernetes_namespace, __meta_kubernetes_service_name, __meta_kubernetes_endpoint_port_name]
          action: keep
          regex: default;kubernetes;https
      - job_name: 'kubernetes-nodes'
        scheme: https
        tls_config:
          ca_file: /var/run/secrets/kubernetes.io/serviceaccount/ca.crt
          insecure_skip_verify: true
        bearer_token_file: /var/run/secrets/kubernetes.io/serviceaccount/token
        kubernetes_sd_configs:
        - role: node
        relabel_configs:
        - action: labelmap
          regex: __meta_kubernetes_node_label_(.+)
        - target_label: __address__
          replacement: kubernetes.default.svc:443
        - source_labels: [__meta_kubernetes_node_name]
          regex: (.+)
          target_label: __metrics_path__
          replacement: /api/v1/nodes/${1}/proxy/metrics
      - job_name: 'kubernetes-pods'
        kubernetes_sd_configs:
        - role: pod
        relabel_configs:
        - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_scrape]
          action: keep
          regex: true
        - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_path]
          action: replace
          target_label: __metrics_path__
          regex: (.+)
        - source_labels: [__address__, __meta_kubernetes_pod_annotation_prometheus_io_port]
          action: replace
          regex: ([^:]+)(?::\\d+)?;(\\d+)
          replacement: $1:$2
          target_label: __address__
        - action: labelmap
          regex: __meta_kubernetes_pod_label_(.+)
        - source_labels: [__meta_kubernetes_namespace]
          action: replace
          target_label: kubernetes_namespace
        - source_labels: [__meta_kubernetes_pod_name]
          action: replace
          target_label: kubernetes_pod_name
      - job_name: 'kubernetes-cadvisor'
        scheme: https
        tls_config:
          ca_file: /var/run/secrets/kubernetes.io/serviceaccount/ca.crt
          insecure_skip_verify: true
        bearer_token_file: /var/run/secrets/kubernetes.io/serviceaccount/token
        kubernetes_sd_configs:
        - role: node
        relabel_configs:
        - action: labelmap
          regex: __meta_kubernetes_node_label_(.+)
        - target_label: __address__
          replacement: kubernetes.default.svc:443
        - source_labels: [__meta_kubernetes_node_name]
          regex: (.+)
          target_label: __metrics_path__
          replacement: /api/v1/nodes/${1}/proxy/metrics/cadvisor
'''
write_yaml('/tmp/configmap-prometheus.yaml', configmap_prometheus)
run('kubectl apply -f /tmp/configmap-prometheus.yaml')


  Sequence 6: ConfigMaps
  ✍️  wrote /tmp/configmap-prometheus.yaml (129 lines)
$ kubectl apply -f /tmp/configmap-prometheus.yaml
------------------------------------------------------------
configmap/prometheus-server-conf created


('configmap/prometheus-server-conf created', 0)

### Step 6.2 — Alertmanager ConfigMap

**Route tree:**
```
All alerts ──▶ default: slack
  ├── severity=info     ──▶ slack
  └── severity=critical ──▶ email  (group_wait=30s, repeat=5m)
```

> ⚠️ Update `SLACK_WEBHOOK_URL`, `TO_EMAIL`, `FROM_EMAIL`, and SMTP settings before production use.

In [11]:
# Sequence 6, Step 2: Alertmanager ConfigMap
# UPDATE THESE VALUES for your environment:
SLACK_WEBHOOK_URL = 'https://hooks.slack.com/services/YOUR/SLACK/WEBHOOK'
TO_EMAIL          = 'alerts@yourdomain.com'
FROM_EMAIL        = 'alertmanager@yourdomain.com'
SMTP_HOST         = 'smtp.yourdomain.com:587'
SMTP_USER         = 'smtpuser'
SMTP_PASS         = 'smtppassword'

# Build config.yml as a clean Python string.
# Note: slack_api_url is a direct child of global: with 2-space indent.
# Do NOT wrap the URL in extra quotes — that caused the
# 'yaml: line 3: could not find expected :' CrashLoopBackOff bug.
config_yml = (
    'global:\n'
    f'  slack_api_url: {SLACK_WEBHOOK_URL}\n'
    'templates:\n'
    "- '/etc/alertmanager/*.tmpl'\n"
    'route:\n'
    "  group_by: ['alertname']\n"
    '  group_wait: 10s\n'
    '  group_interval: 10s\n'
    '  repeat_interval: 86400s\n'
    '  receiver: slack\n'
    '  routes:\n'
    '  - match:\n'
    '      severity: info\n'
    '    receiver: slack\n'
    '  - match:\n'
    '      severity: critical\n'
    '    receiver: email\n'
    '    group_wait: 30s\n'
    '    group_interval: 5m\n'
    '    repeat_interval: 5m\n'
    'receivers:\n'
    '- name: slack\n'
    '  slack_configs:\n'
    '  - send_resolved: true\n'
    "    channel: '#alerts'\n"
    '    username: AlertManager\n'
    '    title: \'{{ template "slack.default.title" . }}\'\n'
    '    text: >-\n'
    '      {{ range .Alerts }}\n'
    '        *Alert:* {{ .Annotations.summary }} - `{{ .Labels.severity }}`\n'
    '        *Description:* {{ .Annotations.description }}\n'
    '        {{ range .Labels.SortedPairs }} - *{{ .Name }}:* `{{ .Value }}`\n'
    '        {{ end }}\n'
    '      {{ end }}\n'
    '- name: email\n'
    '  email_configs:\n'
    f'  - to: {TO_EMAIL}\n'
    f'    from: {FROM_EMAIL}\n'
    f'    smarthost: {SMTP_HOST}\n'
    f'    auth_username: {SMTP_USER}\n'
    f'    auth_password: {SMTP_PASS}\n'
    'inhibit_rules:\n'
    '- source_match:\n'
    '    severity: page\n'
    '  target_match:\n'
    '    severity: page\n'
    "  equal: ['alertname']\n"
)

# Validate YAML before applying
import subprocess as _sp
_r = _sp.run(['python3','-c',f'import yaml; yaml.safe_load({repr(config_yml)}); print("YAML OK")'],
             capture_output=True, text=True)
print(f'  Pre-flight YAML check: {_r.stdout.strip() or _r.stderr.strip()}')

# Wrap in ConfigMap envelope
cm_header = (
    '---\n'
    'kind: ConfigMap\n'
    'apiVersion: v1\n'
    'metadata:\n'
    '  name: alertmanager-config\n'
    '  namespace: monitoring\n'
    'data:\n'
    '  config.yml: |-\n'
)
indented = '\n'.join('    ' + ln if ln.strip() else '' for ln in config_yml.splitlines())
cm_am = cm_header + indented + '\n'

write_yaml('/tmp/configmap-alertmanager.yaml', cm_am)
run('kubectl apply -f /tmp/configmap-alertmanager.yaml')

  Pre-flight YAML check: YAML OK
  ✍️  wrote /tmp/configmap-alertmanager.yaml (55 lines)
$ kubectl apply -f /tmp/configmap-alertmanager.yaml
------------------------------------------------------------
configmap/alertmanager-config created


('configmap/alertmanager-config created', 0)

In [12]:
# Sequence 6, Step 3: Alertmanager notification templates
cm_am_tmpl = r'''---
apiVersion: v1
kind: ConfigMap
metadata:
  name: alertmanager-templates
  namespace: monitoring
data:
  default.tmpl: |
    {{ define "__alertmanager" }}AlertManager{{ end }}
    {{ define "__alertmanagerURL" }}{{ .ExternalURL }}/#/alerts?receiver={{ .Receiver }}{{ end }}
    {{ define "__subject" }}[{{ .Status | toUpper }}{{ if eq .Status "firing" }}:{{ .Alerts.Firing | len }}{{ end }}] {{ .GroupLabels.SortedPairs.Values | join " " }}{{ end }}
    {{ define "__text_alert_list" }}{{ range . }}Labels:
    {{ range .Labels.SortedPairs }} - {{ .Name }} = {{ .Value }}
    {{ end }}Annotations:
    {{ range .Annotations.SortedPairs }} - {{ .Name }} = {{ .Value }}
    {{ end }}Source: {{ .GeneratorURL }}
    {{ end }}{{ end }}
    {{ define "slack.default.title" }}{{ template "__subject" . }}{{ end }}
    {{ define "slack.default.username" }}{{ template "__alertmanager" . }}{{ end }}
    {{ define "slack.default.titlelink" }}{{ template "__alertmanagerURL" . }}{{ end }}
    {{ define "slack.default.text" }}{{ end }}
    {{ define "email.default.subject" }}{{ template "__subject" . }}{{ end }}
    {{ define "pagerduty.default.description" }}{{ template "__subject" . }}{{ end }}
    {{ define "pagerduty.default.client" }}{{ template "__alertmanager" . }}{{ end }}
    {{ define "pagerduty.default.clientURL" }}{{ template "__alertmanagerURL" . }}{{ end }}
    {{ define "pagerduty.default.instances" }}{{ template "__text_alert_list" . }}{{ end }}
    {{ define "opsgenie.default.message" }}{{ template "__subject" . }}{{ end }}
    {{ define "victorops.default.message" }}{{ template "__subject" . }} | {{ template "__alertmanagerURL" . }}{{ end }}
    {{ define "pushover.default.title" }}{{ template "__subject" . }}{{ end }}
    {{ define "pushover.default.url" }}{{ template "__alertmanagerURL" . }}{{ end }}
  slack.tmpl: |
    {{ define "slack.devops.text" }}
    {{range .Alerts}}{{.Annotations.DESCRIPTION}}
    {{end}}
    {{ end }}
'''
write_yaml('/tmp/configmap-alertmanager-tmpl.yaml', cm_am_tmpl)
run('kubectl apply -f /tmp/configmap-alertmanager-tmpl.yaml')

  ✍️  wrote /tmp/configmap-alertmanager-tmpl.yaml (35 lines)
$ kubectl apply -f /tmp/configmap-alertmanager-tmpl.yaml
------------------------------------------------------------
configmap/alertmanager-templates created


('configmap/alertmanager-templates created', 0)

In [13]:
# Sequence 6, Step 4: Grafana datasource ConfigMap
# URL uses Kubernetes internal DNS — Grafana reaches Prometheus without needing NODE_IP
cm_grafana = '''apiVersion: v1
kind: ConfigMap
metadata:
  name: grafana-datasources
  namespace: monitoring
data:
  prometheus.yaml: |-
    {
        "apiVersion": 1,
        "datasources": [
            {
               "access": "proxy",
                "editable": true,
                "name": "prometheus",
                "orgId": 1,
                "type": "prometheus",
                "url": "http://prometheus-service.monitoring.svc:9090",
                "version": 1
            }
        ]
    }
'''
write_yaml('/tmp/configmap-grafana.yaml', cm_grafana)
run('kubectl apply -f /tmp/configmap-grafana.yaml')

print()
print('📋 All ConfigMaps in monitoring namespace:')
run('kubectl get configmap -n monitoring')

  ✍️  wrote /tmp/configmap-grafana.yaml (21 lines)
$ kubectl apply -f /tmp/configmap-grafana.yaml
------------------------------------------------------------
configmap/grafana-datasources created

📋 All ConfigMaps in monitoring namespace:
$ kubectl get configmap -n monitoring
------------------------------------------------------------
NAME                     DATA   AGE
alertmanager-config      1      1s
alertmanager-templates   2      1s
grafana-datasources      1      1s
kube-root-ca.crt         1      65s
prometheus-server-conf   2      1s


('NAME                     DATA   AGE\nalertmanager-config      1      1s\nalertmanager-templates   2      1s\ngrafana-datasources      1      1s\nkube-root-ca.crt         1      65s\nprometheus-server-conf   2      1s',
 0)

---
## Sequence 7 — Deployments

Deploy all three components. The Prometheus deployment automatically uses
`emptyDir` if the NFS PVC is not bound (from Sequence 4b).

| Pod | Image | Storage |
|-----|-------|---------|
| Prometheus | `prom/prometheus` | NFS PVC (or emptyDir fallback) |
| Alertmanager | `prom/alertmanager:v0.19.0` | emptyDir |
| Grafana | `grafana/grafana:latest` | emptyDir |

In [14]:
# Sequence 7, Step 1: Deploy Prometheus
# Automatically selects NFS PVC or emptyDir based on Sequence 4b result
section('Sequence 7: Deployments')

if USE_EMPTY_DIR:
    storage_volume = (
        '        - name: prometheus-storage-volume\n'
        '          emptyDir: {}\n'
    )
    print('  ℹ️  Using emptyDir for Prometheus storage (NFS PVC was not bound)')
else:
    storage_volume = (
        '        - name: prometheus-storage-volume\n'
        '          persistentVolumeClaim:\n'
        '            claimName: pvc-nfs-data\n'
    )
    print(f'  ✅ Using NFS PVC for Prometheus storage (server: {NODE_IP})')

deploy_prom = (
    '---\n'
    'apiVersion: apps/v1\n'
    'kind: Deployment\n'
    'metadata:\n'
    '  name: prometheus-deployment\n'
    '  namespace: monitoring\n'
    '  labels:\n'
    '    app: prometheus-server\n'
    'spec:\n'
    '  replicas: 1\n'
    '  selector:\n'
    '    matchLabels:\n'
    '      app: prometheus-server\n'
    '  template:\n'
    '    metadata:\n'
    '      labels:\n'
    '        app: prometheus-server\n'
    '    spec:\n'
    '      containers:\n'
    '        - name: prometheus\n'
    '          image: prom/prometheus\n'
    '          args:\n'
    '            - "--config.file=/etc/prometheus/prometheus.yml"\n'
    '            - "--storage.tsdb.path=/prometheus/"\n'
    '          ports:\n'
    '            - containerPort: 9090\n'
    '          volumeMounts:\n'
    '            - name: prometheus-config-volume\n'
    '              mountPath: /etc/prometheus/\n'
    '            - name: prometheus-storage-volume\n'
    '              mountPath: /prometheus/\n'
    '      volumes:\n'
    '        - name: prometheus-config-volume\n'
    '          configMap:\n'
    '            defaultMode: 420\n'
    '            name: prometheus-server-conf\n'
) + storage_volume

write_yaml('/tmp/deployment-prometheus.yaml', deploy_prom)
run('kubectl apply -f /tmp/deployment-prometheus.yaml')


  Sequence 7: Deployments
  ℹ️  Using emptyDir for Prometheus storage (NFS PVC was not bound)
  ✍️  wrote /tmp/deployment-prometheus.yaml (38 lines)
$ kubectl apply -f /tmp/deployment-prometheus.yaml
------------------------------------------------------------
deployment.apps/prometheus-deployment created


('deployment.apps/prometheus-deployment created', 0)

In [15]:
# Sequence 7, Step 2: Deploy Alertmanager
deploy_am = '''---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: alertmanager
  namespace: monitoring
spec:
  replicas: 1
  selector:
    matchLabels:
      app: alertmanager
  template:
    metadata:
      name: alertmanager
      labels:
        app: alertmanager
    spec:
      containers:
      - name: alertmanager
        image: prom/alertmanager:v0.19.0
        args:
          - "--config.file=/etc/alertmanager/config.yml"
          - "--storage.path=/alertmanager"
        ports:
        - name: alertmanager
          containerPort: 9093
        volumeMounts:
        - name: config-volume
          mountPath: /etc/alertmanager
        - name: templates-volume
          mountPath: /etc/alertmanager-templates
        - name: alertmanager
          mountPath: /alertmanager
      volumes:
      - name: config-volume
        configMap:
          name: alertmanager-config
      - name: templates-volume
        configMap:
          name: alertmanager-templates
      - name: alertmanager
        emptyDir: {}
'''
write_yaml('/tmp/deployment-alertmanager.yaml', deploy_am)
run('kubectl apply -f /tmp/deployment-alertmanager.yaml')

  ✍️  wrote /tmp/deployment-alertmanager.yaml (42 lines)
$ kubectl apply -f /tmp/deployment-alertmanager.yaml
------------------------------------------------------------
deployment.apps/alertmanager created


('deployment.apps/alertmanager created', 0)

In [16]:
# Sequence 7, Step 3: Deploy Grafana
deploy_grafana = '''apiVersion: apps/v1
kind: Deployment
metadata:
  name: grafana
  namespace: monitoring
spec:
  replicas: 1
  selector:
    matchLabels:
      app: grafana
  template:
    metadata:
      name: grafana
      labels:
        app: grafana
    spec:
      containers:
      - name: grafana
        image: grafana/grafana:latest
        ports:
        - name: grafana
          containerPort: 3000
        resources:
          limits:
            memory: "1Gi"
            cpu: "1000m"
          requests:
            memory: 500M
            cpu: "500m"
        volumeMounts:
          - mountPath: /var/lib/grafana
            name: grafana-storage
          - mountPath: /etc/grafana/provisioning/datasources
            name: grafana-datasources
            readOnly: false
      volumes:
        - name: grafana-storage
          emptyDir: {}
        - name: grafana-datasources
          configMap:
              defaultMode: 420
              name: grafana-datasources
'''
write_yaml('/tmp/deployment-grafana.yaml', deploy_grafana)
run('kubectl apply -f /tmp/deployment-grafana.yaml')

section('Waiting for all deployments to roll out')
for dep in ['prometheus-deployment', 'alertmanager', 'grafana']:
    print(f'  Waiting for {dep}...')
    run(f'kubectl rollout status deployment/{dep} -n monitoring --timeout=180s')

  ✍️  wrote /tmp/deployment-grafana.yaml (42 lines)
$ kubectl apply -f /tmp/deployment-grafana.yaml
------------------------------------------------------------
deployment.apps/grafana created

  Waiting for all deployments to roll out
  Waiting for prometheus-deployment...
$ kubectl rollout status deployment/prometheus-deployment -n monitoring --timeout=180s
------------------------------------------------------------
Waiting for deployment "prometheus-deployment" rollout to finish: 0 of 1 updated replicas are available...
deployment "prometheus-deployment" successfully rolled out
  Waiting for alertmanager...
$ kubectl rollout status deployment/alertmanager -n monitoring --timeout=180s
------------------------------------------------------------
deployment "alertmanager" successfully rolled out
  Waiting for grafana...
$ kubectl rollout status deployment/grafana -n monitoring --timeout=180s
------------------------------------------------------------
deployment "grafana" successfully

---
## Sequence 8 — Services

Expose all three components via LoadBalancer services with fixed NodePorts.

| Service | Port | NodePort | Access URL |
|---------|------|----------|------------|
| `prometheus-service` | 9090 | 30000 | `http://<NODE_IP>:30000` |
| `alertmanager` | 9093 | 31000 | `http://<NODE_IP>:31000` |
| `grafana` | 3000 | 32000 | `http://<NODE_IP>:32000` |

In [17]:
# Sequence 8: Create all three services
section('Sequence 8: Services')

services_yaml = '''apiVersion: v1
kind: Service
metadata:
  name: prometheus-service
  namespace: monitoring
  annotations:
      prometheus.io/scrape: 'true'
      prometheus.io/port:   '9090'
spec:
  selector:
    app: prometheus-server
  type: LoadBalancer
  ports:
    - port: 9090
      targetPort: 9090
      nodePort: 30000
---
apiVersion: v1
kind: Service
metadata:
  name: alertmanager
  namespace: monitoring
  annotations:
      prometheus.io/scrape: 'true'
      prometheus.io/path:   /
      prometheus.io/port:   '8080'
spec:
  selector:
    app: alertmanager
  type: LoadBalancer
  ports:
    - port: 9093
      targetPort: 9093
      nodePort: 31000
---
apiVersion: v1
kind: Service
metadata:
  name: grafana
  namespace: monitoring
  annotations:
      prometheus.io/scrape: 'true'
      prometheus.io/port:   '3000'
spec:
  selector:
    app: grafana
  type: LoadBalancer
  ports:
    - port: 3000
      targetPort: 3000
      nodePort: 32000
'''
write_yaml('/tmp/services.yaml', services_yaml)
run('kubectl apply -f /tmp/services.yaml')

print()
run('kubectl get svc -n monitoring')


  Sequence 8: Services
  ✍️  wrote /tmp/services.yaml (51 lines)
$ kubectl apply -f /tmp/services.yaml
------------------------------------------------------------
service/prometheus-service created
service/alertmanager created
service/grafana created

$ kubectl get svc -n monitoring
------------------------------------------------------------
NAME                 TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE
alertmanager         LoadBalancer   10.43.238.134   <pending>     9093:31000/TCP   0s
grafana              LoadBalancer   10.43.223.118   <pending>     3000:32000/TCP   0s
prometheus-service   LoadBalancer   10.43.112.74    <pending>     9090:30000/TCP   0s


('NAME                 TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE\nalertmanager         LoadBalancer   10.43.238.134   <pending>     9093:31000/TCP   0s\ngrafana              LoadBalancer   10.43.223.118   <pending>     3000:32000/TCP   0s\nprometheus-service   LoadBalancer   10.43.112.74    <pending>     9090:30000/TCP   0s',
 0)

---
## Sequence 9 — Verify the Full Monitoring Stack

**Lab instruction:** Ensure all elements run properly:
```bash
kubectl get all -n monitoring
```

All pods must show `1/1 Running` before accessing the UIs.

In [18]:
# Sequence 9, Step 1: Full stack status check
section('Sequence 9: Verify monitoring stack')
run('kubectl get all -n monitoring')


  Sequence 9: Verify monitoring stack
$ kubectl get all -n monitoring
------------------------------------------------------------
NAME                                         READY   STATUS    RESTARTS   AGE
pod/alertmanager-69ffb5f98-zzvb5             1/1     Running   0          7s
pod/grafana-5f9587668c-pq6nm                 1/1     Running   0          7s
pod/prometheus-deployment-84f65c89c5-nkdlg   1/1     Running   0          7s

NAME                         TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE
service/alertmanager         LoadBalancer   10.43.238.134   <pending>     9093:31000/TCP   0s
service/grafana              LoadBalancer   10.43.223.118   <pending>     3000:32000/TCP   0s
service/prometheus-service   LoadBalancer   10.43.112.74    <pending>     9090:30000/TCP   0s

NAME                                    READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/alertmanager            1/1     1            1           7s
deployment.apps/grafana    

('NAME                                         READY   STATUS    RESTARTS   AGE\npod/alertmanager-69ffb5f98-zzvb5             1/1     Running   0          7s\npod/grafana-5f9587668c-pq6nm                 1/1     Running   0          7s\npod/prometheus-deployment-84f65c89c5-nkdlg   1/1     Running   0          7s\n\nNAME                         TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE\nservice/alertmanager         LoadBalancer   10.43.238.134   <pending>     9093:31000/TCP   0s\nservice/grafana              LoadBalancer   10.43.223.118   <pending>     3000:32000/TCP   0s\nservice/prometheus-service   LoadBalancer   10.43.112.74    <pending>     9090:30000/TCP   0s\n\nNAME                                    READY   UP-TO-DATE   AVAILABLE   AGE\ndeployment.apps/alertmanager            1/1     1            1           7s\ndeployment.apps/grafana                 1/1     1            1           7s\ndeployment.apps/prometheus-deployment   1/1     1            1      

In [19]:
# Sequence 9, Step 2: Print all access URLs (using auto-detected NODE_IP)
section('Access URLs')
print(f'  📊 Prometheus    : http://{NODE_IP}:30000')
print(f'  📊 Targets       : http://{NODE_IP}:30000/targets')
print(f'  📊 Alert Rules   : http://{NODE_IP}:30000/alerts')
print(f'  🔔 Alertmanager  : http://{NODE_IP}:31000')
print(f'  📈 Grafana       : http://{NODE_IP}:32000')
print()
print('  Grafana credentials: admin / admin (change on first login)')
print()
run('kubectl get pods -n monitoring -o wide')


  Access URLs
  📊 Prometheus    : http://10.0.0.95:30000
  📊 Targets       : http://10.0.0.95:30000/targets
  📊 Alert Rules   : http://10.0.0.95:30000/alerts
  🔔 Alertmanager  : http://10.0.0.95:31000
  📈 Grafana       : http://10.0.0.95:32000

  Grafana credentials: admin / admin (change on first login)

$ kubectl get pods -n monitoring -o wide
------------------------------------------------------------
NAME                                     READY   STATUS    RESTARTS   AGE   IP           NODE         NOMINATED NODE   READINESS GATES
alertmanager-69ffb5f98-zzvb5             1/1     Running   0          7s    10.42.0.35   aiops-labs   <none>           <none>
grafana-5f9587668c-pq6nm                 1/1     Running   0          7s    10.42.0.36   aiops-labs   <none>           <none>
prometheus-deployment-84f65c89c5-nkdlg   1/1     Running   0          7s    10.42.0.34   aiops-labs   <none>           <none>


('NAME                                     READY   STATUS    RESTARTS   AGE   IP           NODE         NOMINATED NODE   READINESS GATES\nalertmanager-69ffb5f98-zzvb5             1/1     Running   0          7s    10.42.0.35   aiops-labs   <none>           <none>\ngrafana-5f9587668c-pq6nm                 1/1     Running   0          7s    10.42.0.36   aiops-labs   <none>           <none>\nprometheus-deployment-84f65c89c5-nkdlg   1/1     Running   0          7s    10.42.0.34   aiops-labs   <none>           <none>',
 0)

### Step 9.3 — Live API Verification via Port-Forward

The cells below use `kubectl port-forward` to query each component's
HTTP API directly from this notebook, confirming they are healthy and
correctly configured — without needing to open a browser.

In [20]:
# Sequence 9, Step 3a: Verify Prometheus targets and alert rules
pf1 = subprocess.Popen(
    'kubectl port-forward svc/prometheus-service 9091:9090 -n monitoring',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)

section('Prometheus API Check')
try:
    # Targets
    with urllib.request.urlopen('http://localhost:9091/api/v1/targets', timeout=6) as r:
        data = _json.loads(r.read())
    active = data.get('data',{}).get('activeTargets',[])
    up     = [t for t in active if t.get('health')=='up']
    down   = [t for t in active if t.get('health')!='up']
    print(f'  ✅ Prometheus reachable')
    print(f'  Active targets: {len(active)}  |  UP: {len(up)}  |  DOWN: {len(down)}')
    print()
    print('  Target health:')
    for t in active:
        job    = t.get('labels',{}).get('job','?')
        health = t.get('health','?')
        inst   = t.get('labels',{}).get('instance','?')[:45]
        icon   = '🟢' if health=='up' else '🔴'
        print(f'    {icon} [{job}] {inst}')

    # Alert rules
    with urllib.request.urlopen('http://localhost:9091/api/v1/rules', timeout=6) as r:
        rdata = _json.loads(r.read())
    groups = rdata.get('data',{}).get('groups',[])
    rules  = [rule for g in groups for rule in g.get('rules',[])]
    print()
    print(f'  Alert rules loaded: {len(rules)}')
    for rule in rules:
        state = rule.get('state','?')
        icon  = '🔴' if state=='firing' else '🟡' if state=='pending' else '✅'
        print(f'    {icon} {rule.get("name","?")} [{state}]')

except Exception as e:
    print(f'  ❌ Could not reach Prometheus: {e}')
    print(f'     Access manually: http://{NODE_IP}:30000/targets')
finally:
    pf1.terminate()


  Prometheus API Check
  ✅ Prometheus reachable
  Active targets: 0  |  UP: 0  |  DOWN: 0

  Target health:

  Alert rules loaded: 4
    ✅ High Pod Memory [inactive]
    ✅ Cluster Memory Usage [inactive]
    ✅ Cluster CPU Usage [inactive]
    ✅ Pod Memory usage [inactive]


In [21]:
# Sequence 9, Step 3b: Verify Alertmanager
pf2 = subprocess.Popen(
    'kubectl port-forward svc/alertmanager 9094:9093 -n monitoring',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)

section('Alertmanager API Check')
try:
    with urllib.request.urlopen('http://localhost:9094/api/v1/status', timeout=6) as r:
        data = _json.loads(r.read())
    cfg = data.get('data',{})
    ver = cfg.get('versionInfo',{}).get('version','?')
    print(f'  ✅ Alertmanager reachable')
    print(f'  Version: {ver}')

    with urllib.request.urlopen('http://localhost:9094/api/v1/alerts', timeout=6) as r:
        adata = _json.loads(r.read())
    alerts = adata.get('data',[])
    print(f'  Active alerts: {len(alerts)}')
    if alerts:
        for a in alerts[:5]:
            name = a.get('labels',{}).get('alertname','?')
            sev  = a.get('labels',{}).get('severity','?')
            print(f'    🔴 {name} [{sev}]')
    else:
        print('  ✅ No active alerts')
except Exception as e:
    print(f'  ❌ Could not reach Alertmanager: {e}')
    print(f'     Access manually: http://{NODE_IP}:31000')
finally:
    pf2.terminate()


  Alertmanager API Check
  ✅ Alertmanager reachable
  Version: 0.19.0
  Active alerts: 0
  ✅ No active alerts


In [22]:
# Sequence 9, Step 3c: Verify Grafana
pf3 = subprocess.Popen(
    'kubectl port-forward svc/grafana 3001:3000 -n monitoring',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)

section('Grafana API Check')
try:
    with urllib.request.urlopen('http://localhost:3001/api/health', timeout=6) as r:
        data = _json.loads(r.read())
    gver = data.get('version','?')
    gdb  = data.get('database','?')
    print(f'  ✅ Grafana reachable')
    print(f'  Version:  {gver}')
    print(f'  Database: {gdb}')
    print()
    print(f'  Access Grafana at: http://{NODE_IP}:32000')
    print('  Default credentials: admin / admin')
    print()
    print('  After login:')
    print('  1. Configuration → Data Sources → prometheus → Test')
    print('  2. Dashboards → Import → ID 315 (K8s cluster monitoring)')
except Exception as e:
    print(f'  ❌ Could not reach Grafana: {e}')
    print(f'     Access manually: http://{NODE_IP}:32000')
finally:
    pf3.terminate()


  Grafana API Check
  ✅ Grafana reachable
  Version:  13.0.0
  Database: ok

  Access Grafana at: http://10.0.0.95:32000
  Default credentials: admin / admin

  After login:
  1. Configuration → Data Sources → prometheus → Test
  2. Dashboards → Import → ID 315 (K8s cluster monitoring)


### Step 9.4 — NFS Storage Final Verification

This cell confirms Prometheus is actually writing data to the NFS volume
(or reports that emptyDir is in use).

In [23]:
# Sequence 9, Step 4: Confirm Prometheus is writing to storage
section('Storage verification')

if USE_EMPTY_DIR:
    print('  ℹ️  Prometheus is using emptyDir (non-persistent)')
    print('  To switch to NFS:')
    print('    1. Fix NFS: sudo systemctl start nfs-server && sudo exportfs -ra')
    print('    2. Re-run Sequence 2 and 4')
    print('    3. Re-run Sequence 7 (deployment will auto-select NFS PVC)')
else:
    print(f'  Checking NFS data directory: {NFS_EXPORT_DIR}')
    run(f'ls -la {NFS_EXPORT_DIR}/')
    print()
    # Check PVC is still bound
    pvc_phase, _ = run_q('kubectl get pvc pvc-nfs-data -n monitoring -o jsonpath="{.status.phase}"')
    if pvc_phase == 'Bound':
        print('  ✅ PVC is Bound — Prometheus is persisting metrics to NFS')
    else:
        print(f'  ⚠️  PVC phase: {pvc_phase}')


  Storage verification
  ℹ️  Prometheus is using emptyDir (non-persistent)
  To switch to NFS:
    1. Fix NFS: sudo systemctl start nfs-server && sudo exportfs -ra
    2. Re-run Sequence 2 and 4
    3. Re-run Sequence 7 (deployment will auto-select NFS PVC)


---
## 🔧 Troubleshooting Guide

| Symptom | Command to run | Likely cause |
|---------|---------------|-------------|
| `ContainerCreating` for Prometheus | `kubectl describe pod <name> -n monitoring` | NFS not mounted |
| `CrashLoopBackOff` for Alertmanager | `kubectl logs <name> -n monitoring --previous` | YAML error in config.yml |
| PVC stuck `Pending` | `kubectl describe pvc pvc-nfs-data -n monitoring` | NFS server down or wrong IP |
| Prometheus targets all DOWN | `kubectl logs deploy/prometheus-deployment -n monitoring` | RBAC missing |
| Grafana blank/no data | Check datasource URL in Grafana UI | Wrong Prometheus service URL |

In [24]:
# Troubleshooting: Full status + recent events
section('Pod Status')
run('kubectl get pods -n monitoring -o wide')

section('Recent Events (last 15)')
run('kubectl get events -n monitoring --sort-by=.lastTimestamp 2>/dev/null | tail -15')

section('PV / PVC Status')
run('kubectl get pv,pvc -n monitoring')


  Pod Status
$ kubectl get pods -n monitoring -o wide
------------------------------------------------------------
NAME                                     READY   STATUS    RESTARTS   AGE   IP           NODE         NOMINATED NODE   READINESS GATES
alertmanager-69ffb5f98-zzvb5             1/1     Running   0          19s   10.42.0.35   aiops-labs   <none>           <none>
grafana-5f9587668c-pq6nm                 1/1     Running   0          19s   10.42.0.36   aiops-labs   <none>           <none>
prometheus-deployment-84f65c89c5-nkdlg   1/1     Running   0          19s   10.42.0.34   aiops-labs   <none>           <none>

  Recent Events (last 15)
$ kubectl get events -n monitoring --sort-by=.lastTimestamp 2>/dev/null | tail -15
------------------------------------------------------------
18s         Normal    Started              pod/grafana-5f9587668c-pq6nm                  Started container grafana
18s         Normal    Created              pod/grafana-5f9587668c-pq6nm              

('NAME                           CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS     CLAIM                     STORAGECLASS   REASON   AGE\npersistentvolume/pv-nfs-data   1Gi        RWX            Retain           Released   monitoring/pvc-nfs-data   managed-nfs             27m\n\nNAME                                 STATUS    VOLUME   CAPACITY   ACCESS MODES   STORAGECLASS   AGE\npersistentvolumeclaim/pvc-nfs-data   Pending                                      managed-nfs    82s',
 0)

In [25]:
# Troubleshooting: Component logs
section('Prometheus Logs (last 30 lines)')
run('kubectl logs -n monitoring deploy/prometheus-deployment --tail=30 2>/dev/null')

section('Alertmanager Logs (last 20 lines)')
run('kubectl logs -n monitoring deploy/alertmanager --tail=20 2>/dev/null')

section('Grafana Logs (last 20 lines)')
run('kubectl logs -n monitoring deploy/grafana --tail=20 2>/dev/null')


  Prometheus Logs (last 30 lines)
$ kubectl logs -n monitoring deploy/prometheus-deployment --tail=30 2>/dev/null
------------------------------------------------------------
time=2026-04-16T04:08:33.767Z level=INFO source=main.go:1678 msg="updated GOGC" old=100 new=75
time=2026-04-16T04:08:33.768Z level=INFO source=main.go:744 msg="Leaving GOMAXPROCS=12: CPU quota undefined" component=automaxprocs
time=2026-04-16T04:08:33.768Z level=INFO source=memlimit.go:198 msg="GOMEMLIMIT is updated" component=automemlimit package=github.com/KimMachineGun/automemlimit/memlimit GOMEMLIMIT=45024086016 previous=9223372036854775807
time=2026-04-16T04:08:33.768Z level=INFO source=main.go:851 msg="Starting Prometheus Server" mode=server version="(version=3.11.2, branch=HEAD, revision=f0f0fdd679dcd6df320b0558b20919f7cd44c407)"
time=2026-04-16T04:08:33.768Z level=INFO source=main.go:856 msg="operational information" build_context="(go=go1.26.2, platform=linux/amd64, user=root@d2684485c347, date=20260413-

('logger=plugin.angulardetectorsprovider.dynamic t=2026-04-16T04:08:37.450193977Z level=info msg="Patterns update finished" duration=79.440965ms\nlogger=plugin.installer t=2026-04-16T04:08:37.631212038Z level=info msg="Installing plugin" pluginId=grafana-lokiexplore-app version=\nlogger=installer.fs t=2026-04-16T04:08:37.761383106Z level=info msg="Downloaded and extracted grafana-lokiexplore-app v2.0.3 zip successfully to /var/lib/grafana/plugins/grafana-lokiexplore-app"\nlogger=plugins.registration t=2026-04-16T04:08:37.793283063Z level=info msg="Plugin registered" pluginId=grafana-lokiexplore-app\nlogger=plugin.backgroundinstaller t=2026-04-16T04:08:37.793321776Z level=info msg="Plugin successfully installed" pluginId=grafana-lokiexplore-app version= duration=603.685112ms\nlogger=plugin.backgroundinstaller t=2026-04-16T04:08:37.793345512Z level=info msg="Installing plugin" pluginId=grafana-pyroscope-app version=\nlogger=plugin.installer t=2026-04-16T04:08:38.05120989Z level=info msg=

In [26]:
# Troubleshooting: NFS server status
section('NFS Server Status')
run('systemctl status nfs-server --no-pager')

section('NFS Exports')
run('sudo exportfs -v')

section('showmount')
run(f'showmount -e {NODE_IP}')

section('NFS Data Directory')
run(f'ls -la {NFS_EXPORT_DIR}/')


  NFS Server Status
$ systemctl status nfs-server --no-pager
------------------------------------------------------------
● nfs-server.service - NFS server and services
     Loaded: loaded (/usr/lib/systemd/system/nfs-server.service; enabled; preset: disabled)
    Drop-In: /run/systemd/generator/nfs-server.service.d
             └─order-with-mounts.conf
     Active: active (exited) since Thu 2026-04-16 03:46:08 GMT; 22min ago
       Docs: man:rpc.nfsd(8)
             man:exportfs(8)
   Main PID: 586761 (code=exited, status=0/SUCCESS)
        CPU: 31ms

Apr 16 03:46:07 aiops-labs systemd[1]: Starting NFS server and services...
Apr 16 03:46:08 aiops-labs systemd[1]: Finished NFS server and services.

  NFS Exports
$ sudo exportfs -v
------------------------------------------------------------
/mnt/nfs/promdata
		10.10.0.100(sync,wdelay,hide,no_subtree_check,sec=sys,rw,secure,root_squash,no_all_squash)

  showmount
$ showmount -e 10.0.0.95
------------------------------------------------

('total 0\ndrwxrwxrwx 2 nobody nobody  6 Apr 16 03:46 .\ndrwxr-xr-x 3 root   root   22 Apr 16 03:46 ..',
 0)

In [ ]:
# Troubleshooting: Describe a specific pod
# Update POD_NAME to the actual pod from: kubectl get pods -n monitoring
POD_NAME = 'prometheus-deployment-xxx-yyy'  # <-- update this

# Uncomment to run:
# run(f'kubectl describe pod {POD_NAME} -n monitoring')
# run(f'kubectl logs {POD_NAME} -n monitoring --previous')

print('Update POD_NAME above and uncomment the run() lines to describe/inspect a pod.')

---
## 🧹 Cleanup

Removes all Kubernetes resources and optionally stops the NFS server.

> ⚠️ This deletes all metrics history and Grafana dashboards.
> Only run when you have finished the lab.

In [ ]:
# Cleanup — uncomment lines below

# Remove all K8s resources
# run('kubectl delete namespace monitoring --wait=false')
# run('kubectl delete clusterrole prometheus kube-state-metrics', ignore_errors=True)
# run('kubectl delete clusterrolebinding prometheus kube-state-metrics', ignore_errors=True)
# run('kubectl delete pv pv-nfs-data', ignore_errors=True)

# Optionally stop NFS (leave exported if you want to reuse it)
# run('sudo systemctl stop nfs-server')
# run('sudo systemctl disable nfs-server')

# Verify K8s cleanup
# run('kubectl get namespace monitoring 2>/dev/null || echo deleted')

print('Cleanup is commented out by default to prevent accidental deletion.')
print('Uncomment the lines above when ready to tear down.')

---
## 📚 Lab Recap

### What We Built

```
 Rocky Linux 8.7 Node
 ├── NFS Server
 │   ├── nfs-utils installed and enabled
 │   ├── /mnt/nfs/promdata (nobody:nobody, 777)
 │   ├── /etc/exports configured
 │   └── firewall ports open (nfs, mountd, rpc-bind)
 │
 └── monitoring namespace
     ├── RBAC
     │   ├── ClusterRole: prometheus + kube-state-metrics
     │   ├── ServiceAccount: prometheus
     │   └── ClusterRoleBindings: prometheus + kube-state-metrics
     ├── Storage
     │   ├── PersistentVolume: pv-nfs-data (NFS 1Gi)
     │   └── PersistentVolumeClaim: pvc-nfs-data (500Mi)
     ├── ConfigMaps
     │   ├── prometheus-server-conf (prometheus.yml + 4 alert rules)
     │   ├── alertmanager-config    (Slack + email routing)
     │   ├── alertmanager-templates (Go notification templates)
     │   └── grafana-datasources    (auto-provision Prometheus)
     ├── Deployments
     │   ├── prometheus-deployment  (prom/prometheus)
     │   ├── alertmanager           (prom/alertmanager:v0.19.0)
     │   └── grafana                (grafana/grafana:latest)
     └── Services (LoadBalancer)
         ├── prometheus-service     :9090 → NodePort 30000
         ├── alertmanager           :9093 → NodePort 31000
         └── grafana                :3000 → NodePort 32000
```

---

### Key Concepts Covered

| Concept | What You Learned |
|---------|------------------|
| **NFS setup** | Install, configure, export, firewall, verify with showmount |
| **Auto IP detection** | Find non-localhost IP via kubectl, interface scan, or UDP socket |
| **PVC binding check** | Wait loop with clear pass/fail and emptyDir fallback |
| **YAML validation** | Pre-flight `yaml.safe_load` before applying ConfigMaps |
| **Alertmanager YAML** | Correct `global.slack_api_url` indentation to avoid CrashLoopBackOff |
| **Prometheus scrape jobs** | 5 jobs: self, apiservers, nodes, pods, cAdvisor |
| **Service discovery** | `prometheus.io/scrape: true` annotation enables auto-discovery |
| **Grafana provisioning** | Auto-configure datasource via ConfigMap mount |

---

*Lab 3a v2 complete — Congratulations! 🎉*